In [ ]:
# ═══════════════════════════════════════════════════════════════
# 0. 환경 설정 — 한글 폰트
# ═══════════════════════════════════════════════════════════════
!apt-get install -y fonts-nanum > /dev/null 2>&1

import glob, re, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

paths = glob.glob('/usr/share/fonts/**/Nanum*.ttf', recursive=True)
if not paths:
    !wget -q -O /tmp/NanumGothic.ttf https://github.com/googlefonts/nanum-gothic/raw/main/fonts/ttf/NanumGothic-Regular.ttf
    paths = ['/tmp/NanumGothic.ttf']
for p in paths:
    fm.fontManager.addfont(p)
plt.rcParams['font.family'] = fm.FontProperties(fname=paths[0]).get_name()
plt.rcParams['axes.unicode_minus'] = False
print('폰트:', plt.rcParams['font.family'])

BASE = '/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)'
KEEP_Q = 0.01                                # 약한 필터 — 축소가 소표본을 처리하므로 하한만
# 2024·2025 는 기록지가 미완성이라 시즌 스탯(v2)에서 뺐다(2026-08-06) - 학습에서도 뺀다.
TRAIN_COHORTS  = [2020, 2021, 2022, 2023]
PREDICT_COHORT = 2026
ALIAS = {'동성고': '광주동성고'}              # 원본 오타/약칭 보정 (게광고는 원본에서 수정됨)

def norm(x):
    """학교명 정규화 — 접미사가 겹쳐 붙은 경우까지 반복 제거.
       예: '아산BC(U-18)' → '아산',  '아산BC' → '아산'  (양쪽이 같아야 조인됨)"""
    x = re.sub(r'\s+', '', str(x))
    prev = None
    while prev != x:
        prev = x
        x = re.sub(r'(\(U-?1[89]\)|BC|SBC|AC|SC|야구단)$', '', x)
    return x.replace('공업고', '공고').replace('상업고', '상고')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1. 로드 + 지명 레이블 판정
#    레이블은 항상 드래프트 원본에서 새로 만든다 (중간 파일을 신뢰하지 않음).
#    고졸 지명 = 드래프트 당시 소속이 '고교 리그에서 뛴 팀'인 경우.
#    팀 이름 규칙이 아니라 실제 명단/기록에 있는지로 판정한다
#    (용인시야구단·아산BC 같은 U-18 클럽팀도 고졸 지명 대상이므로).
# ═══════════════════════════════════════════════════════════════
BAT = pd.read_excel(f'{BASE}/batter_stats_all_v2.xlsx')
PIT = pd.read_excel(f'{BASE}/pitcher_stats_all_v2.xlsx')
PL  = pd.concat([pd.read_excel(f'{BASE}/players_by_season/players_{y}.xlsx')
                 for y in [2023, 2024, 2025, 2026]], ignore_index=True)

for d in (BAT, PIT): d['학교_n'] = d['학교'].apply(norm)
PL['학교_n'] = PL['학교명'].apply(norm)
hs_teams = set(BAT['학교_n']) | set(PIT['학교_n']) | set(PL['학교_n'])

# 드래프트 원본 → 레이블
sheets = pd.read_excel(f'{BASE}/KBO 신인드래프트 지명자.xlsx', sheet_name=None)
LAB = pd.concat([s.assign(draft_year=int(re.search(r'\d{4}', n).group()))
                 for n, s in sheets.items()], ignore_index=True).dropna(subset=['이름'])
LAB['이름_n'] = LAB['이름'].astype(str).str.strip()
LAB['학교_n'] = LAB['학교'].apply(norm).replace(ALIAS)
LAB['지명']   = LAB['학교_n'].isin(hs_teams).astype(int)
LAB['cohort'] = LAB['draft_year']
LAB.to_excel(f'{BASE}/draft_labels.xlsx', index=False)

print(f'고교 팀 {len(hs_teams)}개 / 전체 지명자 {len(LAB)}명 / 고졸 {LAB["지명"].sum()}명')
print(LAB.groupby('draft_year')['지명'].agg(고졸='sum', 전체='size').to_string())
print('\n[고졸 아님 — 대학/독립/해외가 맞는지 확인]')
print(LAB[LAB['지명'] == 0]['학교'].value_counts().to_string())


# ── 해외(MLB) 직행 선수 ─────────────────────────────────────────
#   KBO 드래프트에 신청하지 않으므로 위 레이블에서는 '미지명'이 된다. 그런데 이들은
#   최상위 유망주라(장현석 $1.0M, 김성준 $1.2M) 음성으로 두면 모델을 거꾸로 가르친다.
#   그래서 타깃을 'KBO 지명'이 아니라 '프로 진출'로 보고 양성으로 처리한다.
#   원본에 포지션이 없어 로스터 → 기록 순으로 역추적한다.
FGN = pd.concat(
    [s for s in pd.read_excel(f'{BASE}/해외_입단_2020_2029.xlsx', sheet_name=None).values()],
    ignore_index=True).dropna(subset=['이름'])
FGN['이름_n'] = FGN['이름'].astype(str).str.strip()
FGN['학교_n'] = FGN['고교'].apply(norm)
FGN['연도']   = pd.to_numeric(FGN['연도'], errors='coerce').astype('Int64')

# 포지션 결정 순서: ① 원본의 포지션 열 → ② 로스터 → ③ 기록량 추론
roster_pos = (PL[PL['포지션'].astype(str).str.strip().ne('미지정')]
              .groupby(['성명','학교_n','연도'])['포지션'].first())
bat_pa = BAT.groupby(['이름','학교_n','year'])['PA'].sum()
pit_ip = PIT.groupby(['이름','학교_n','year'])['IP'].sum()

def split_pos(p):
    """포지션 문자열 → (투수인가, 타자인가). '투타겸업'에는 '투수'가 문자로 없다."""
    p = str(p)
    겸업 = '투타겸업' in p
    return ('투수' in p) or 겸업, any(t in p for t in ('내야','외야','포수','타자')) or 겸업

rows = []
for _, r in FGN.iterrows():
    key = (r['이름_n'], r['학교_n'], int(r['연도']))
    pa, ip = bat_pa.get(key, 0), pit_ip.get(key, 0)
    src_pos = str(r.get('포지션', '')).strip()

    if src_pos and src_pos.lower() not in ('nan', '미지정'):
        is_p, is_b = split_pos(src_pos)
        src = f'원본({src_pos})'
    else:
        p = roster_pos.get(key)
        if isinstance(p, str) and p.strip():
            is_p, is_b = split_pos(p)
            src = f'로스터({p})'
        else:
            is_p, is_b, src = ip >= 20, pa >= 80, '기록추론'
    if not (is_p or is_b):                   # 둘 다 아니면 기록이 많은 쪽으로
        is_p, is_b = ip > 0 and ip >= pa / 4, pa > 0
        src += '·보정'
    rows.append({**r.to_dict(), 'P_투수': bool(is_p), 'P_타자': bool(is_b),
                 'PA': pa, 'IP': round(float(ip), 1), '판정근거': src})
FGN = pd.DataFrame(rows)

print(f'\n[해외 진출자 {len(FGN)}명 — 포지션 역추적]')
print(FGN[['연도','이름_n','고교','입단 구단','PA','IP','P_타자','P_투수','판정근거']]
      .to_string(index=False))


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2. 레이블 조인 + 코호트 배정
#    포지션으로 갈라 붙인다: 내야수로 지명된 선수가 어쩌다 몇 이닝 던졌다고
#    투수 모델에서 '지명'이 되면 안 된다(안재석 2020 1R 1순위가 그 사례).
#    투타겸업/포수겸투수 표기는 양쪽에 모두 들어간다 — 실제로 양쪽으로 평가받은 선수이므로.
# ═══════════════════════════════════════════════════════════════
pos = LAB['포지션'].astype(str)
겸업 = pos.str.contains('투타겸업')          # '투타겸업'에는 '투수'가 문자로 없다 — 따로 잡아야 함
LAB['P_투수'] = pos.str.contains('투수') | 겸업
LAB['P_타자'] = pos.str.contains('내야|외야|포수|타자') | 겸업
print('[지명자 포지션 분해]')
print(f"  투수 {int((LAB['지명'] & LAB['P_투수']).sum())}명 / "
      f"타자 {int((LAB['지명'] & LAB['P_타자']).sum())}명 "
      f"(겸업 {int((LAB['지명'] & LAB['P_투수'] & LAB['P_타자']).sum())}명 중복)")

대졸 = set(LAB[LAB['지명'] == 0]['이름_n'])      # 대학·독립·해외 지명자

def attach(d, role):
    z = (LAB[(LAB['지명'] == 1) & LAB[role]]
         [['이름_n','학교_n','draft_year','라운드','순위','포지션','지명구단']]
         .drop_duplicates(subset=['이름_n','학교_n','draft_year']))
    m = d.merge(z, left_on=['이름','학교_n','year'],
                right_on=['이름_n','학교_n','draft_year'], how='left')

    # 해외 직행도 '프로 진출'이므로 양성. 포지션이 맞는 쪽에만 붙인다.
    해외키 = set(zip(FGN.loc[FGN[role], '이름_n'],
                    FGN.loc[FGN[role], '학교_n'],
                    FGN.loc[FGN[role], '연도'].astype(int)))
    m['해외진출'] = [k in 해외키 for k in zip(m['이름'], m['학교_n'], m['year'])]

    # 해외 진출자는 draft_year가 없다. 계약 연도가 곧 3학년 연도이고 매칭도
    # year 기준으로 했으니, 그 값을 draft_year에 채워 코호트가 확정되게 한다.
    m.loc[m['해외진출'] & m['draft_year'].isna(), 'draft_year'] =         m.loc[m['해외진출'] & m['draft_year'].isna(), 'year']

    m['지명']          = (m['라운드'].notna() | m['해외진출']).astype(int)
    m['해외진출']      = m['해외진출'].astype(int)
    m['대학진학후지명'] = m['이름'].isin(대졸).astype(int)
    return m

BAT, PIT = attach(BAT, 'P_타자'), attach(PIT, 'P_투수')

# 학년 → 코호트 (players 파일이 있는 2023~2026)
학년맵 = (PL.dropna(subset=['학년'])
            .assign(cohort=lambda x: x['연도'] + (3 - x['학년']))
            .groupby(['성명','학교_n','연도'])['cohort'].first())

def cohort(d):
    d = d.copy()
    d['cohort'] = np.nan
    # ① 지명자 — 드래프트 연도로 확정
    d.loc[d['지명'] == 1, 'cohort'] = d.loc[d['지명'] == 1, 'draft_year']
    # ② 학년 기반
    d['_c'] = pd.Series(list(zip(d['이름'], d['학교_n'], d['year'])),
                        index=d.index).map(학년맵)
    d['cohort'] = d['cohort'].fillna(d['_c'])
    # ③ 선수 단위 전파 (코호트는 선수의 고정 속성)
    prop = d.dropna(subset=['cohort']).groupby(['이름','학교_n'])['cohort'].min()
    d['cohort'] = d['cohort'].fillna(
        pd.Series(list(zip(d['이름'], d['학교_n'])), index=d.index).map(prop))
    # ④ 남은 선수 — 마지막 관측 시즌을 3학년으로 추정
    last = d.groupby(['이름','학교_n'])['year'].max()
    d['cohort_추정'] = d['cohort'].isna().astype(int)
    d['cohort'] = d['cohort'].fillna(
        pd.Series(list(zip(d['이름'], d['학교_n'])), index=d.index).map(last))
    return d.drop(columns=['_c'])

BAT, PIT = cohort(BAT), cohort(PIT)

print('\n[코호트 추정 의존도]  ※ 지명자는 0%여야 정상')
for 라벨, d in [('타자', BAT), ('투수', PIT)]:
    print(f'  {라벨}: 전체 {d["cohort_추정"].mean()*100:.1f}% / '
          f'지명자 {d[d.지명==1]["cohort_추정"].mean()*100:.1f}%')

print('[해외 진출자 반영]')
for 라벨, d in [('타자', BAT), ('투수', PIT)]:
    n = d[(d['해외진출'] == 1) & (d['year'] == d['cohort'])]['이름'].nunique() if 'cohort' in d else d['해외진출'].sum()
    print(f'  {라벨}: 양성으로 추가된 해외 진출자 {int(d["해외진출"].sum())}행')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3. 3학년 추출 → (약한) 필터 → 평균으로의 축소 → 코호트 내 백분위
#
#   고교야구는 대회 토너먼트라 팀이 일찍 탈락하면 경기 수가 반토막 난다.
#   그래서 타석/이닝이 적은 건 '실력이 없어서'가 아니라 '팀이 일찍 떨어져서'인 경우가 많다.
#   실제로 엄격한 필터를 걸었더니 1라운드 지명자 5명(이병헌 2순위, 신범준·김건희 6순위,
#   정민규 9순위, 손성빈 10순위)이 통째로 빠졌다.
#
#   그래서 버리는 대신 축소한다 — 표본이 작으면 리그 평균 쪽으로 당겨서 과대평가를 막고,
#   선수는 데이터에 남긴다. 세이버매트릭스의 표준 처리이기도 하다.
#   원시 카운트(H·HR·BB…) 대신 비율을 쓰고, 기회량은 PA/IP로 따로 넣는다.
# ═══════════════════════════════════════════════════════════════
BAT3 = BAT[BAT['year'] == BAT['cohort']].copy()
PIT3 = PIT[PIT['year'] == PIT['cohort']].copy()
print(f'[3학년 행] 타자 {len(BAT3)} (지명 {BAT3["지명"].sum()}) / '
      f'투수 {len(PIT3)} (지명 {PIT3["지명"].sum()})')

# ── 약한 필터: 축소로 감당 안 되는 극소 표본만 제거 ──
#   타자 50타석은 고정값 — 1라운드 지명자(손성빈 50타석)가 들어오는 선.
#   투수는 지명자 하위 KEEP_Q 지점. 이병헌(2021 1R 2순위)은 2경기 4.3이닝뿐이라
#   여기서 빠지는데, 그 표본으로는 신호가 없어 제외가 타당하다고 봤다.
MIN_PA = 50
MIN_IP = max(3,  PIT3.loc[PIT3['지명'] == 1, 'IP'].quantile(KEEP_Q))
before = (BAT3['지명'].sum(), PIT3['지명'].sum())
BAT3 = BAT3[BAT3['PA'] >= MIN_PA].copy()
PIT3 = PIT3[PIT3['IP'] >= MIN_IP].copy()
print(f'[필터] PA>={MIN_PA:.0f} / IP>={MIN_IP:.1f}  →  '
      f'타자 {len(BAT3)} (지명 -{before[0]-BAT3["지명"].sum()}) / '
      f'투수 {len(PIT3)} (지명 -{before[1]-PIT3["지명"].sum()})')

# ── 파생 비율 ──
BAT3['HR_rate']  = BAT3['HR'] / BAT3['PA']
BAT3['XBH_rate'] = (BAT3['2B'] + BAT3['3B'] + BAT3['HR']) / BAT3['PA']

# ── 평균으로의 축소 ──
def shrink(d, rate, n, k):
    """w = n/(n+k). 표본이 k일 때 리그 평균과 반반 섞인다."""
    out = pd.Series(index=d.index, dtype=float)
    for _, g in d.groupby('cohort'):
        v, w_n = g[rate], g[n]
        ok = v.notna()
        lg = (v[ok] * w_n[ok]).sum() / w_n[ok].sum() if ok.any() else np.nan
        w = w_n / (w_n + k)
        out.loc[g.index] = (w * v.fillna(lg) + (1 - w) * lg)
    return out

K_PA, K_IP = 100, 30          # 반쯤 믿기 시작하는 표본 크기

BAT_RATE = ['AVG','OBP','SLG','OPS','IsoP','IsoD','K%','BB%','BABIP','HR_rate','XBH_rate']
PIT_RATE = ['ERA','WHIP','FIP','BAA','K/9','BB/9','HR/9','K%','BB%','K/BB']

for c in BAT_RATE:
    if c in BAT3.columns: BAT3[c] = shrink(BAT3, c, 'PA', K_PA)
for c in PIT_RATE:
    if c in PIT3.columns: PIT3[c] = shrink(PIT3, c, 'IP', K_IP)
print(f'\n[축소 적용] 타자 {len(BAT_RATE)}개 지표 (k={K_PA}타석) / '
      f'투수 {len(PIT_RATE)}개 (k={K_IP}이닝)')

# ── 코호트 내 백분위 ── (기회량 PA/IP는 축소 없이 그대로 순위화)
def pctize(d, feats):
    for c in feats:
        if c in d.columns:
            d[f'{c}_pct'] = d.groupby('cohort')[c].rank(pct=True) * 100
    return d

BAT3 = pctize(BAT3, BAT_RATE + ['PA'])
PIT3 = pctize(PIT3, PIT_RATE + ['IP'])

BAT3.to_excel(f'{BASE}/model_batter.xlsx', index=False)
PIT3.to_excel(f'{BASE}/model_pitcher.xlsx', index=False)

for 라벨, d in [('타자', BAT3), ('투수', PIT3)]:
    print(f'\n[{라벨}] 지명률 {d["지명"].mean()*100:.1f}% / '
          f'피처 {len([c for c in d.columns if c.endswith("_pct")])}개')
    print(d.groupby('cohort')['지명'].agg(전체='size', 지명='sum').T.to_string())

# ── 축소가 의도대로 작동했는지 ──
print('\n[검증] 필터에서 빠졌던 1라운드 지명자가 남아 있는가')
chk = ['이병헌','신범준','김건희','정민규','손성빈','윤동희']
for 라벨, d, r in [('타자', BAT3, 'OPS'), ('투수', PIT3, 'ERA')]:
    m = d[d['이름'].isin(chk) & (d['지명'] == 1)]
    if len(m):
        print(f'  [{라벨}]')
        print(m[['이름','학교','cohort','PA' if 라벨=='타자' else 'IP', r, f'{r}_pct','라운드']]
              .round(3).to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4. 학습 · 평가 · 2026 예측
#    · 코호트별 leave-one-out — 과거로 학습해 그 해를 예측하는 실사용 시나리오
#    · 임계값 판정 대신 랭킹(Precision@K) 중심 — 참고자료로 쓰는 용도에 맞춤
#    · 양성이 코호트당 40~50명뿐이라 모델은 얕게
# ═══════════════════════════════════════════════════════════════
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve

PARAMS = dict(n_estimators=300, learning_rate=0.05, max_depth=3, num_leaves=7,
              min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
              reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)

def precision_at_k(y, p, k):
    return int(np.asarray(y)[np.argsort(p)[::-1][:k]].sum())

def run(path, 라벨):
    d = pd.read_excel(path)
    F = [c for c in d.columns if c.endswith('_pct')]
    tr = d[d.cohort.isin(TRAIN_COHORTS)].reset_index(drop=True)
    te = d[d.cohort == PREDICT_COHORT].reset_index(drop=True)
    print(f'\n{"="*58}\n[{라벨}] 학습 {len(tr)}행(지명 {tr["지명"].sum()}) / '
          f'예측 {len(te)}행 / 피처 {len(F)}개')

    # ── 코호트별 leave-one-out ──
    oof = np.full(len(tr), np.nan)
    for c in TRAIN_COHORTS:
        m = (tr.cohort != c).values
        mdl = lgb.LGBMClassifier(**PARAMS).fit(tr.loc[m, F], tr.loc[m, '지명'])
        oof[~m] = mdl.predict_proba(tr.loc[~m, F])[:, 1]
    tr['pred'] = oof

    print(f'\n전체 AUROC {roc_auc_score(tr["지명"], oof):.4f}')
    rows = [{'코호트': c, 'n': len(s), '지명': int(s['지명'].sum()),
             'AUROC': roc_auc_score(s['지명'], s['pred']),
             'P@10': precision_at_k(s['지명'], s['pred'], 10),
             'P@30': precision_at_k(s['지명'], s['pred'], 30),
             'P@50': precision_at_k(s['지명'], s['pred'], 50)}
            for c, s in tr.groupby('cohort')]
    print(pd.DataFrame(rows).round(3).to_string(index=False))

    # ── 캘리브레이션 ──
    pt, pp = calibration_curve(tr['지명'], oof, n_bins=10, strategy='quantile')
    fig, ax = plt.subplots(figsize=(4.4, 4.4))
    ax.plot([0, 1], [0, 1], '--', c='gray', lw=1)
    ax.plot(pp, pt, 'o-', c='#378ADD')
    ax.set_xlabel('예측 확률'); ax.set_ylabel('실제 지명률')
    ax.set_title(f'{라벨} — 캘리브레이션'); ax.grid(alpha=.25)
    plt.tight_layout(); plt.show()

    # ── 오분류 = 발견 ──
    lo = tr[tr.지명 == 0].nlargest(15, 'pred')[['이름','학교','cohort','pred','대학진학후지명']]
    print(f'\n[저평가 — 기록 상위인데 미지명 TOP 15]')
    print(lo.round(3).to_string(index=False))
    print(f'  → 이 중 대학·독립 경로로 지명된 선수: {int(lo["대학진학후지명"].sum())}명')
    print(f'\n[서프라이즈 — 기록 하위인데 지명 TOP 15]')
    print(tr[tr.지명 == 1].nsmallest(15, 'pred')
          [['이름','학교','cohort','pred','라운드','순위']].round(3).to_string(index=False))

    # ── 최종 학습 → 2026 예측 ──
    final = lgb.LGBMClassifier(**PARAMS).fit(tr[F], tr['지명'])
    te = te.copy(); te['pred'] = final.predict_proba(te[F])[:, 1]
    # 이미 해외 구단과 계약한 선수는 계약 완료 시점에 KBO 드래프트 신청 자격이 없다.
    # 예측 명단에서 빼되, 모델이 몇 위로 봤는지는 남긴다 — 검증할 때
    # "모델이 최상위로 짚은 선수가 MLB로 갔다"는 설명이 되기 때문.
    te['_rank_all'] = te['pred'].rank(ascending=False, method='min').astype(int)
    gone = te[te['해외진출'] == 1]
    if len(gone):
        print('[해외 계약 — 드래프트 신청 불가, 예측 명단에서 제외]')
        print(gone.sort_values('pred', ascending=False)
              [['이름', '학교', 'pred', '_rank_all']]
              .rename(columns={'_rank_all': '제외전_순위'}).round(3).to_string(index=False))
    te = te[te['해외진출'] == 0].copy()

    top = te.nlargest(50, 'pred')[['이름','학교','pred']].reset_index(drop=True)
    top.index += 1
    print(f'\n[2026 예측 상위 25]  (해외 계약자 제외, 대상 {len(te)}명)')
    print(top.head(25).round(3).to_string())

    imp = pd.Series(final.feature_importances_, index=F).nlargest(12)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(imp.index[::-1], imp.values[::-1], color='#1D9E75')
    ax.set_title(f'{라벨} — 피처 중요도'); ax.spines[['top','right']].set_visible(False)
    plt.tight_layout(); plt.show()

    te.sort_values('pred', ascending=False).to_excel(
        f'{BASE}/pred_2026_{라벨}.xlsx', index=False)
    joblib.dump({'model': final, 'features': F}, f'{BASE}/model_{라벨}.pkl')
    return tr, te

bat_tr, bat_te = run(f'{BASE}/model_batter.xlsx',  '타자')
pit_tr, pit_te = run(f'{BASE}/model_pitcher.xlsx', '투수')
print('\n저장: pred_2026_타자.xlsx / pred_2026_투수.xlsx / model_*.pkl')

In [ ]:
BAT3_all = BAT[BAT['year'] == BAT['cohort']]
PIT3_all = PIT[PIT['year'] == PIT['cohort']]

print('[표본 부족으로 제외된 지명자 — 타자]')
print(BAT3_all[(BAT3_all['지명']==1) & (BAT3_all['PA'] < MIN_PA)]
      [['이름','학교','cohort','PA','G','라운드','순위']]
      .sort_values('순위').to_string(index=False))

print('\n[표본 부족으로 제외된 지명자 — 투수]')
print(PIT3_all[(PIT3_all['지명']==1) & (PIT3_all['IP'] < MIN_IP)]
      [['이름','학교','cohort','IP','G','라운드','순위']]
      .sort_values('순위').to_string(index=False))


In [ ]:
from datetime import datetime
stamp = '2026-07-20'          # 오늘 날짜 고정

for 라벨 in ['타자','투수']:
    d = pd.read_excel(f'{BASE}/pred_2026_{라벨}.xlsx')
    d.to_excel(f'{BASE}/SEALED_{stamp}_pred_2026_{라벨}.xlsx', index=False)
print(f'봉인 완료 — SEALED_{stamp}_*')
